# 04. 하이퍼파라미터 탐색

**목표**
- 실험 1: LoRA rank (4, 8, 16, 32) 비교
- 실험 2: 학습률 (1e-4, 5e-5, 1e-5) 비교
- 실험 3: LoRA target modules 비교

**전략**: 각 실험은 epoch=3으로 빠르게 비교 (Kaggle 30시간 제한 고려)

**실행 환경**: Kaggle (T4/P100)

In [ ]:
# 셀 1 — 레포 클론
!rm -rf /kaggle/working/korean-doc-understanding
!git clone https://github.com/shimtaehun/korean-doc-understanding.git

In [ ]:
# 셀 2 — 경로 추가
import sys
sys.path.append("/kaggle/working/korean-doc-understanding")

In [ ]:
# 셀 3 — 패키지 설치
!pip install -q "transformers==4.41.0" "tokenizers==0.19.1" peft accelerate datasets wandb jiwer scikit-learn albumentations pyyaml

In [ ]:
# 셀 4 — flash_attn mock
import sys
import types
import importlib.machinery
import transformers.utils.import_utils

for mod_name in ['flash_attn', 'flash_attn.flash_attn_interface', 'flash_attn.bert_padding']:
    mock = types.ModuleType(mod_name)
    mock.__spec__ = importlib.machinery.ModuleSpec(mod_name, loader=None)
    sys.modules[mod_name] = mock

transformers.utils.import_utils.is_flash_attn_2_available = lambda: False
print("flash_attn mock 완료")

In [ ]:
# 셀 5 — imports + GPU 확인
import os
import torch
import wandb
import yaml
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup

from src.data.dataset import CORDDataset
from src.model.florence_lora import LoRASettings, load_florence_with_lora
from src.training.evaluate import evaluate
from src.training.callbacks import WandBCallback, CheckpointCallback

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 셀 6 — WandB 로그인
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=api_key, relogin=True)

In [ ]:
# 셀 7 — 기본 config 로드
with open("/kaggle/working/korean-doc-understanding/configs/train_config.yaml") as f:
    base_cfg = yaml.safe_load(f)

# 공통 설정 (실험마다 고정)
base_cfg["output"]["checkpoint_dir"] = "/kaggle/working/checkpoints"
base_cfg["wandb"]["entity"]           = "sthun0211-home"
base_cfg["training"]["batch_size"]    = 2
base_cfg["training"]["gradient_accumulation_steps"] = 8
base_cfg["training"]["epochs"]        = 3  # 빠른 비교를 위해 3 epoch

# 데이터셋은 한 번만 로드
hf_dataset = load_dataset(base_cfg["data"]["hf_dataset"])
print(f"train: {len(hf_dataset['train'])}개 / val: {len(hf_dataset['validation'])}개")

## 공통 학습 함수

In [ ]:
# 셀 8 — 실험 1회 실행 함수
import copy

def run_experiment(exp_name: str, lora_settings: LoRASettings, cfg: dict) -> dict:
    """실험 1회 실행 후 최종 val 메트릭 반환."""
    print(f"\n{'='*50}")
    print(f"실험: {exp_name}")
    print(f"{'='*50}")

    # 모델 로드
    model, processor = load_florence_with_lora(
        model_id=cfg["model"]["name"],
        lora_settings=lora_settings,
        device=DEVICE,
    )

    # DataLoader
    train_ds = CORDDataset(hf_dataset["train"], processor, split="train",
                           max_length=cfg["model"]["max_length"], augment=True)
    val_ds   = CORDDataset(hf_dataset["validation"], processor, split="validation",
                           max_length=cfg["model"]["max_length"], augment=False)
    train_loader = DataLoader(train_ds, batch_size=cfg["training"]["batch_size"],
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg["training"]["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)

    # WandB run
    run = wandb.init(
        project=cfg["wandb"]["project"],
        entity=cfg["wandb"]["entity"],
        name=exp_name,
        config={
            "lora_r": lora_settings.r,
            "lora_alpha": lora_settings.alpha,
            "lora_target_modules": lora_settings.target_modules,
            "lr": cfg["training"]["learning_rate"],
            "epochs": cfg["training"]["epochs"],
            "batch_size": cfg["training"]["batch_size"],
            "grad_accum": cfg["training"]["gradient_accumulation_steps"],
        },
        reinit=True,
    )

    # Optimizer + Scheduler
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["training"]["learning_rate"],
        weight_decay=cfg["training"]["weight_decay"],
    )
    total_steps  = len(train_loader) * cfg["training"]["epochs"]
    warmup_steps = int(total_steps * cfg["training"]["warmup_ratio"])
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = GradScaler(enabled=cfg["training"]["fp16"])
    accum_steps  = cfg["training"]["gradient_accumulation_steps"]

    best_f1 = 0.0

    for epoch in range(1, cfg["training"]["epochs"] + 1):
        model.train()
        total_loss = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            pixel_values   = batch["pixel_values"].to(DEVICE)
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            with autocast(dtype=torch.float16, enabled=cfg["training"]["fp16"]):
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                loss = outputs.loss / accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["training"]["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item() * accum_steps

        val_metrics = evaluate(model, processor, val_loader, DEVICE)
        epoch_loss  = total_loss / len(train_loader)
        best_f1     = max(best_f1, val_metrics["field_f1"])

        print(f"  Epoch {epoch} | loss={epoch_loss:.4f} | f1={val_metrics['field_f1']:.4f} | cer={val_metrics['cer']:.4f}")
        wandb.log({"train/loss": epoch_loss, "val/field_f1": val_metrics["field_f1"],
                   "val/cer": val_metrics["cer"], "epoch": epoch})

    wandb.summary["best_field_f1"] = best_f1
    wandb.finish()

    # 메모리 해제
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

    return {"exp_name": exp_name, "best_f1": best_f1}

print("run_experiment 함수 정의 완료")

## 실험 1: LoRA Rank 비교

**가설**: rank가 높을수록 성능은 올라가지만 수익 체감이 있을 것.  
rank=8 또는 16에서 최적일 것으로 예상.

In [ ]:
# 셀 9 — 실험 1: rank 비교 (lr=5e-5 고정)
rank_results = []

for rank in [4, 8, 16, 32]:
    cfg = copy.deepcopy(base_cfg)
    lora = LoRASettings(
        r=rank,
        alpha=rank * 2,  # alpha = rank * 2 관례
        dropout=0.05,
        target_modules=["q_proj", "v_proj"],
    )
    result = run_experiment(f"rank-{rank}", lora, cfg)
    rank_results.append(result)

print("\n=== 실험 1 결과 ===")
for r in rank_results:
    print(f"  {r['exp_name']}: best F1 = {r['best_f1']:.4f}")

best_rank = max(rank_results, key=lambda x: x["best_f1"])["exp_name"]
print(f"\n→ 최적 rank: {best_rank}")

## 실험 2: 학습률 비교

**가설**: 너무 높으면 발산, 너무 낮으면 수렴 느림.  
Florence-2 파인튜닝 레퍼런스 기준 5e-5가 최적일 것으로 예상.

In [ ]:
# 셀 10 — 실험 2: lr 비교 (rank는 실험 1 최적값 사용)
# 실험 1 결과에서 최적 rank 수동 입력
BEST_RANK = 8  # ← 실험 1 결과 보고 수정

lr_results = []

for lr in [1e-4, 5e-5, 1e-5]:
    cfg = copy.deepcopy(base_cfg)
    cfg["training"]["learning_rate"] = lr
    lora = LoRASettings(
        r=BEST_RANK,
        alpha=BEST_RANK * 2,
        dropout=0.05,
        target_modules=["q_proj", "v_proj"],
    )
    result = run_experiment(f"lr-{lr:.0e}", lora, cfg)
    lr_results.append(result)

print("\n=== 실험 2 결과 ===")
for r in lr_results:
    print(f"  {r['exp_name']}: best F1 = {r['best_f1']:.4f}")

best_lr = max(lr_results, key=lambda x: x["best_f1"])["exp_name"]
print(f"\n→ 최적 lr: {best_lr}")

## 실험 3: LoRA Target Modules 비교

**가설**: attention + FFN 전체에 LoRA를 적용하면 성능이 오르지만 파라미터도 증가할 것.

In [ ]:
# 셀 11 — 실험 3: target modules 비교
# 실험 1, 2 결과에서 최적값 수동 입력
BEST_RANK = 8   # ← 실험 1 결과 보고 수정
BEST_LR   = 5e-5  # ← 실험 2 결과 보고 수정

module_experiments = [
    ("attn-qv",      ["q_proj", "v_proj"]),
    ("attn-full",    ["q_proj", "k_proj", "v_proj", "o_proj"]),
    ("attn-ffn",     ["q_proj", "v_proj", "fc1", "fc2"]),
]

module_results = []

for exp_name, target_modules in module_experiments:
    cfg = copy.deepcopy(base_cfg)
    cfg["training"]["learning_rate"] = BEST_LR
    lora = LoRASettings(
        r=BEST_RANK,
        alpha=BEST_RANK * 2,
        dropout=0.05,
        target_modules=target_modules,
    )
    result = run_experiment(exp_name, lora, cfg)
    module_results.append(result)

print("\n=== 실험 3 결과 ===")
for r in module_results:
    print(f"  {r['exp_name']}: best F1 = {r['best_f1']:.4f}")

## 최종 결과 정리

In [ ]:
# 셀 12 — 전체 실험 결과 요약 출력
import pandas as pd

all_results = rank_results + lr_results + module_results
df = pd.DataFrame(all_results).sort_values("best_f1", ascending=False)
print(df.to_string(index=False))

best = df.iloc[0]
print(f"""
=== docs/EXPERIMENTS.md에 기록할 내용 ===

### 실험 1: LoRA rank 비교
{chr(10).join(f'  rank={r["exp_name"]}: F1={r["best_f1"]:.4f}' for r in rank_results)}

### 실험 2: 학습률 비교
{chr(10).join(f'  lr={r["exp_name"]}: F1={r["best_f1"]:.4f}' for r in lr_results)}

### 실험 3: target modules 비교
{chr(10).join(f'  {r["exp_name"]}: F1={r["best_f1"]:.4f}' for r in module_results)}

최적 조합: {best['exp_name']} (F1={best['best_f1']:.4f})
""")